# Proyecto de Minería de Datos
## Modelo de Regresión con Árboles de Decisión

### Proyecto: Predicción del porcentaje de apoyo al equipo local

En este notebook construiremos un **modelo de regresión con Árboles de Decisión**.

La idea es predecir un valor numérico:

> **¿Qué porcentaje de personas apoyará al equipo local antes de un partido?**

### Variable objetivo

`Bet_Perc_on_Home`

Esta variable representa el **porcentaje de apoyo o apuesta hacia el equipo local**.

---

### Diferencia con el trabajo anterior

En el trabajo de clasificación predecíamos una categoría:

- S = gana local
- H = empate
- A = gana visitante

Ahora ya no predecimos una categoría.

En regresión queremos predecir un **número**, por ejemplo:

- 35%
- 52%
- 71%

Por eso necesitamos una variable objetivo numérica.


# 1. Definición del problema

## ¿Qué queremos predecir?

Queremos estimar el porcentaje de apoyo que tendrá el equipo local.

La variable objetivo será:

`Bet_Perc_on_Home`

## ¿Por qué esta variable?

Porque es una variable **numérica continua**, por lo que cumple con el requisito de utilizar regresión.

## Variables que utilizaremos para explicar el porcentaje

Usaremos estadísticas previas de los equipos, como:

- Posición en la liga.
- Partidos jugados.
- Puntos obtenidos.
- Partidos ganados.
- Empates.
- Derrotas.
- Goles anotados.
- Goles recibidos.
- Diferencia de goles.
- Porcentaje histórico de victorias.
- Porcentaje histórico de empates.
- Enfrentamientos entre ambos equipos.

## Importante: evitar fuga de información

No utilizaremos como entradas:

- `Votes_for_Home`
- `Votes_for_Draw`
- `Votes_for_Away`
- `Total_Bettors`

Esto se hace porque estas variables pueden estar directamente relacionadas con el porcentaje que queremos predecir.

Si las usáramos, el modelo podría estar viendo indirectamente la respuesta.

A esto se le llama **data leakage o fuga de información**.

### Para explicar en exposición

> “Nuestro objetivo es predecir el porcentaje de apoyo al equipo local utilizando estadísticas anteriores de los equipos. No usamos los votos directamente porque podrían revelar la respuesta.”


# 2. Importación de librerías

En esta parte cargamos las herramientas que utilizaremos.

- `pandas`: manejo de datos.
- `numpy`: operaciones numéricas.
- `matplotlib`: gráficos.
- `sklearn`: entrenamiento y evaluación del modelo.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_error
)

pd.set_option("display.max_columns", None)

print("Librerías cargadas correctamente.")


# 3. Carga del dataset

Nuestro archivo se encuentra en:

`Data/raw/predicciones_personas.csv`

Como este notebook debe estar dentro de la carpeta `Codigo`, usamos la ruta:

`../Data/raw/predicciones_personas.csv`

El `..` significa “subir una carpeta”.

### Estructura esperada

```text
Grupo_Patitogit/
│
├── Codigo/
│   └── 07_arbol_regresion_explicado.ipynb
│
└── Data/
    └── raw/
        └── predicciones_personas.csv
```


In [ ]:
# Ruta del archivo
ruta = "../Data/raw/predicciones_personas.csv"

# Leer el CSV
datos = pd.read_csv(ruta)

print("Dataset cargado correctamente.")
print("Dimensiones:", datos.shape)

# Mostrar las primeras 5 filas
display(datos.head())


## ¿Qué significa `datos.shape`?

El resultado tiene la forma:

`(filas, columnas)`

Por ejemplo:

`(11417, 50)`

significaría que tenemos 11.417 registros y 50 variables.

### Para explicar

> “Primero cargamos el dataset y verificamos cuántos registros y variables contiene.”


# 4. Revisión de la variable objetivo

Antes de entrenar el modelo debemos verificar que la variable objetivo exista y revisar sus valores.


In [ ]:
objetivo = "Bet_Perc_on_Home"

if objetivo not in datos.columns:
    raise ValueError(
        f"No se encontró la columna {objetivo}. "
        "Revisa el nombre de las columnas del dataset."
    )

print("Variable objetivo encontrada:", objetivo)

print("\nResumen estadístico:")
display(datos[objetivo].describe())

print("\nCantidad de valores nulos:")
print(datos[objetivo].isna().sum())


## ¿Cómo interpretar el resumen?

`describe()` muestra:

- **count:** cantidad de valores disponibles.
- **mean:** promedio.
- **std:** dispersión de los datos.
- **min:** valor mínimo.
- **25%:** primer cuartil.
- **50%:** mediana.
- **75%:** tercer cuartil.
- **max:** valor máximo.

Esto nos ayuda a entender cómo se comporta la variable que queremos predecir.

### Para explicar

> “Revisamos la variable objetivo para comprobar que tenga valores numéricos válidos y conocer su distribución.”


# 5. Selección de variables independientes

Las variables independientes son las características que el modelo utilizará para intentar predecir `Bet_Perc_on_Home`.

Seleccionamos estadísticas deportivas previas al partido.

El código verifica automáticamente cuáles existen realmente en el dataset.


In [ ]:
variables_numericas_candidatas = [
    "League_Rank_1",
    "League_Rank_2",
    "Games_played_1",
    "Games_played_2",
    "Points_1",
    "Points_2",
    "Won_1",
    "Won_2",
    "Draw_1",
    "Draw_2",
    "Lost_1",
    "Lost_2",
    "Goals_Scored_1",
    "Goals_Scored_2",
    "Goals_Rec_1",
    "Goal_Rec_2",
    "Goals_Diff_1",
    "Goals_Diff_2",
    "Win_Perc_1",
    "Win_Perc_2",
    "Draw_Perc_1",
    "Draw_Perc_2",
    "Number_of_H2H_matches"
]

variables_categoricas_candidatas = [
    "Weekday",
    "League_type_country"
]

# Tomar únicamente columnas que sí existan
variables_numericas = [
    c for c in variables_numericas_candidatas
    if c in datos.columns
]

variables_categoricas = [
    c for c in variables_categoricas_candidatas
    if c in datos.columns
]

variables_independientes = (
    variables_numericas + variables_categoricas
)

print("Variables numéricas utilizadas:")
print(variables_numericas)

print("\nVariables categóricas utilizadas:")
print(variables_categoricas)

print("\nTotal de variables utilizadas:")
print(len(variables_independientes))


## ¿Por qué usamos estas variables?

Porque representan el rendimiento previo de ambos equipos.

Por ejemplo:

- Si un equipo tiene más puntos, puede generar mayor confianza.
- Si tiene mejor posición en la liga, puede recibir más apoyo.
- Si marca más goles, también puede influir en la preferencia de las personas.

Esto no significa que estas variables causen directamente las apuestas, pero sí pueden contener información útil para realizar la predicción.

### Para explicar

> “Seleccionamos variables relacionadas con el rendimiento deportivo porque pueden influir en la percepción que las personas tienen de cada equipo.”


# 6. Limpieza y preparación de datos

Aquí realizaremos tres tareas:

1. Eliminar registros donde falte la variable objetivo.
2. Completar valores faltantes en las variables independientes.
3. Convertir variables categóricas a números.

## ¿Por qué?

Los modelos de Machine Learning necesitan trabajar con datos completos y numéricos.


In [ ]:
# Eliminar únicamente filas sin valor objetivo
datos_modelo = datos.dropna(
    subset=["Bet_Perc_on_Home"]
).copy()

# X = variables independientes
X = datos_modelo[variables_independientes]

# y = variable objetivo
y = datos_modelo["Bet_Perc_on_Home"]

print("Cantidad de registros disponibles:", len(datos_modelo))
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)


## Tratamiento de valores faltantes

Para las variables numéricas utilizaremos la **mediana**.

Ejemplo:

Si tenemos:

`10, 12, 15, NaN, 100`

la mediana permite completar el dato sin dejar que un valor extremo como 100 afecte demasiado.

Para variables categóricas utilizamos la **moda**, es decir, el valor que más se repite.


In [ ]:
# Pipeline para variables numéricas
pipeline_numerico = Pipeline(
    steps=[
        (
            "imputador",
            SimpleImputer(strategy="median")
        )
    ]
)

# Lista de transformaciones
transformadores = [
    (
        "numericas",
        pipeline_numerico,
        variables_numericas
    )
]

# Si existen variables categóricas:
if variables_categoricas:

    pipeline_categorico = Pipeline(
        steps=[
            (
                "imputador",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "codificador",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )

    transformadores.append(
        (
            "categoricas",
            pipeline_categorico,
            variables_categoricas
        )
    )

# Unir todas las transformaciones
preprocesador = ColumnTransformer(
    transformers=transformadores
)

print("Preprocesador preparado correctamente.")


## ¿Qué es One-Hot Encoding?

Un modelo no entiende directamente palabras como:

`Monday`, `Tuesday`, `Wednesday`

Por eso las convertimos en columnas numéricas.

Por ejemplo:

```text
Monday    Tuesday
1         0
0         1
```

### Para explicar

> “Los valores faltantes se completaron con mediana o moda y las variables de texto se transformaron a formato numérico.”


# 7. División en entrenamiento y prueba

Dividiremos los datos en:

- **80% entrenamiento**
- **20% prueba**

## ¿Por qué hacemos esto?

El modelo aprende con el conjunto de entrenamiento.

Después lo probamos con datos que no utilizó para aprender.

Así podemos comprobar si realmente aprendió un patrón o simplemente memorizó los datos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)

print("\nPorcentaje entrenamiento:",
      round(len(X_train) / len(X) * 100, 2), "%")

print("Porcentaje prueba:",
      round(len(X_test) / len(X) * 100, 2), "%")


### Para explicar

> “Utilizamos el 80% de los datos para enseñar al modelo y dejamos el 20% para comprobar qué tan bien predice datos que no había visto.”


# 8. Entrenamiento del Árbol de Decisión inicial

Primero entrenaremos un árbol con sus parámetros por defecto.

Este será nuestro **modelo inicial**.

Después construiremos una versión optimizada para comprobar si podemos mejorar su rendimiento.


In [ ]:
modelo_inicial = Pipeline(
    steps=[
        (
            "preprocesador",
            preprocesador
        ),
        (
            "modelo",
            DecisionTreeRegressor(
                random_state=42
            )
        )
    ]
)

# Entrenar
modelo_inicial.fit(
    X_train,
    y_train
)

# Predecir datos de prueba
pred_inicial = modelo_inicial.predict(
    X_test
)

print("Árbol inicial entrenado correctamente.")


# 9. Métricas de regresión

En clasificación utilizábamos:

- Accuracy
- Precision
- Recall
- F1

En regresión estas métricas ya no corresponden.

Ahora utilizamos principalmente:

## MSE

**Mean Squared Error — Error Cuadrático Medio**

Mide cuánto se equivoca el modelo, dando mayor castigo a errores grandes.

> **Mientras menor sea el MSE, mejor.**

---

## R²

Mide qué tanto de la variación de los datos puede explicar el modelo.

En términos simples:

- Cerca de **1** → explica muy bien los datos.
- Cerca de **0** → explica poco.
- Menor que **0** → el modelo está funcionando peor que una predicción básica basada en el promedio.

> **Mientras mayor sea R², mejor.**

---

## MAE

**Mean Absolute Error — Error Absoluto Medio**

Es muy fácil de interpretar.

Si obtenemos:

`MAE = 8`

significa aproximadamente:

> “El modelo se equivoca en promedio por 8 puntos porcentuales.”

Aunque la consigna pide MSE y R², incluimos MAE porque facilita mucho la interpretación.


In [ ]:
def evaluar_regresion(
    y_real,
    y_predicho,
    nombre
):
    # Error cuadrático medio
    mse = mean_squared_error(
        y_real,
        y_predicho
    )

    # Raíz del MSE
    rmse = np.sqrt(mse)

    # Error absoluto medio
    mae = mean_absolute_error(
        y_real,
        y_predicho
    )

    # Coeficiente R²
    r2 = r2_score(
        y_real,
        y_predicho
    )

    print("=" * 45)
    print(nombre)
    print("=" * 45)

    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"R²   : {r2:.4f}")

    return {
        "Modelo": nombre,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }


resultado_inicial = evaluar_regresion(
    y_test,
    pred_inicial,
    "Árbol inicial"
)


## ¿Qué debes mirar aquí?

Principalmente:

- ¿Qué valor tiene el MSE?
- ¿Qué valor tiene el R²?
- ¿Cuántos puntos porcentuales indica el MAE?

Guarda estos valores porque luego los compararemos con el modelo optimizado.


# 10. Ajuste de hiperparámetros

Un Árbol de Decisión puede crecer demasiado y terminar memorizando los datos.

Para evitarlo probaremos distintas configuraciones.

## Hiperparámetros utilizados

### `max_depth`

Controla cuántos niveles puede tener el árbol.

### `criterion`

Indica cómo decide cuál es la mejor división.

### `min_samples_split`

Cantidad mínima de registros necesarios para dividir un nodo.

### `min_samples_leaf`

Cantidad mínima de registros que debe tener una hoja.

## GridSearchCV

`GridSearchCV` prueba automáticamente muchas combinaciones y selecciona la que obtiene mejores resultados mediante validación cruzada.


In [ ]:
modelo_busqueda = Pipeline(
    steps=[
        (
            "preprocesador",
            preprocesador
        ),
        (
            "modelo",
            DecisionTreeRegressor(
                random_state=42
            )
        )
    ]
)

parametros = {
    "modelo__max_depth": [
        3, 5, 7, 10, None
    ],
    "modelo__criterion": [
        "squared_error",
        "friedman_mse"
    ],
    "modelo__min_samples_split": [
        2, 5, 10
    ],
    "modelo__min_samples_leaf": [
        1, 2, 5, 10
    ]
}

busqueda = GridSearchCV(
    estimator=modelo_busqueda,
    param_grid=parametros,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

busqueda.fit(
    X_train,
    y_train
)

print("\nMejores parámetros encontrados:")
print(busqueda.best_params_)

print("\nMSE promedio de validación:")
print(round(-busqueda.best_score_, 4))


### Para explicar

> “En lugar de elegir los parámetros manualmente, utilizamos GridSearchCV para probar diferentes configuraciones y seleccionar automáticamente la que obtuvo menor error.”


# 11. Evaluación del modelo optimizado

Ahora tomamos la mejor configuración encontrada y volvemos a realizar predicciones con el conjunto de prueba.


In [ ]:
modelo_optimo = busqueda.best_estimator_

pred_optimo = modelo_optimo.predict(
    X_test
)

resultado_optimo = evaluar_regresion(
    y_test,
    pred_optimo,
    "Árbol optimizado"
)


# 12. Comparación entre el árbol inicial y el optimizado

Recordemos:

- **MSE menor = mejor**
- **RMSE menor = mejor**
- **MAE menor = mejor**
- **R² mayor = mejor**


In [ ]:
comparacion = pd.DataFrame(
    [
        resultado_inicial,
        resultado_optimo
    ]
).set_index("Modelo")

display(
    comparacion.round(4)
)


In [ ]:
# Interpretación automática de la comparación

mse_inicial = resultado_inicial["MSE"]
mse_optimo = resultado_optimo["MSE"]

r2_inicial = resultado_inicial["R2"]
r2_optimo = resultado_optimo["R2"]

if mse_optimo < mse_inicial:
    print(
        "✅ El modelo optimizado redujo el MSE."
    )
else:
    print(
        "⚠️ El modelo optimizado no redujo el MSE."
    )

if r2_optimo > r2_inicial:
    print(
        "✅ El modelo optimizado mejoró el R²."
    )
else:
    print(
        "⚠️ El modelo optimizado no mejoró el R²."
    )


### Para explicar

> “Comparamos el modelo inicial con el optimizado. Para saber cuál funciona mejor buscamos un MSE más bajo y un R² más alto.”


# 13. Gráfico de valores reales vs valores predichos

Este gráfico compara:

- Eje X: valor real.
- Eje Y: valor predicho por el modelo.

La línea diagonal representa una predicción perfecta.

Si los puntos están cerca de la línea, significa que el modelo está prediciendo valores cercanos a los reales.


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    pred_optimo,
    alpha=0.5
)

minimo = min(
    y_test.min(),
    pred_optimo.min()
)

maximo = max(
    y_test.max(),
    pred_optimo.max()
)

plt.plot(
    [minimo, maximo],
    [minimo, maximo],
    linestyle="--",
    label="Predicción perfecta"
)

plt.xlabel(
    "Porcentaje real de apoyo al local"
)

plt.ylabel(
    "Porcentaje predicho"
)

plt.title(
    "Valores reales vs valores predichos"
)

plt.legend()
plt.tight_layout()
plt.show()


## ¿Cómo interpretar este gráfico?

- Punto cerca de la diagonal → buena predicción.
- Punto lejos de la diagonal → error mayor.
- Muchos puntos cerca de la línea → mejor desempeño general.

### Para explicar

> “Cada punto representa un partido. Mientras más cerca esté de la línea diagonal, más cerca estuvo la predicción del valor real.”


# 14. Importancia de variables

Una ventaja del Árbol de Decisión es que podemos observar qué variables utilizó más para realizar sus predicciones.

Esto se conoce como **feature importance**.

Una importancia alta significa que la variable fue útil para dividir los datos durante el entrenamiento.

Importante:

> Que una variable sea importante para el modelo no significa necesariamente que sea la causa del comportamiento de las personas.


In [ ]:
preprocesador_entrenado = (
    modelo_optimo.named_steps[
        "preprocesador"
    ]
)

arbol = (
    modelo_optimo.named_steps[
        "modelo"
    ]
)

nombres_variables = (
    preprocesador_entrenado
    .get_feature_names_out()
)

importancias = (
    arbol.feature_importances_
)

tabla_importancias = pd.DataFrame({
    "Variable": nombres_variables,
    "Importancia": importancias
}).sort_values(
    "Importancia",
    ascending=False
)

print("Variables más importantes:")

display(
    tabla_importancias.head(15)
)

top = (
    tabla_importancias
    .head(15)
    .sort_values("Importancia")
)

plt.figure(figsize=(10, 7))

plt.barh(
    top["Variable"],
    top["Importancia"]
)

plt.title(
    "Variables más importantes para el modelo"
)

plt.xlabel("Importancia")
plt.ylabel("Variable")

plt.tight_layout()
plt.show()


### Para explicar

> “Este gráfico muestra qué variables utilizó más el Árbol de Decisión. La barra más grande corresponde a la variable con mayor importancia dentro del modelo.”


# 15. Visualización del Árbol de Decisión

El árbol completo puede ser demasiado grande.

Por eso mostraremos únicamente los primeros tres niveles.

Esto **no significa que el modelo esté incompleto**.

Los `(...)` indican que existen más ramas debajo, pero no las mostramos para que el gráfico pueda leerse.


In [ ]:
# Traducir nombres para que el árbol sea más fácil de entender

traducciones = {
    "League_Rank_1": "Posición local",
    "League_Rank_2": "Posición visitante",
    "Games_played_1": "Partidos local",
    "Games_played_2": "Partidos visitante",
    "Points_1": "Puntos local",
    "Points_2": "Puntos visitante",
    "Won_1": "Victorias local",
    "Won_2": "Victorias visitante",
    "Draw_1": "Empates local",
    "Draw_2": "Empates visitante",
    "Lost_1": "Derrotas local",
    "Lost_2": "Derrotas visitante",
    "Goals_Scored_1": "Goles local",
    "Goals_Scored_2": "Goles visitante",
    "Goals_Rec_1": "Goles recibidos local",
    "Goal_Rec_2": "Goles recibidos visitante",
    "Goals_Diff_1": "Dif. goles local",
    "Goals_Diff_2": "Dif. goles visitante",
    "Win_Perc_1": "% victorias local",
    "Win_Perc_2": "% victorias visitante",
    "Draw_Perc_1": "% empates local",
    "Draw_Perc_2": "% empates visitante",
    "Number_of_H2H_matches": "Partidos entre ambos"
}

nombres_limpios = []

for nombre in nombres_variables:

    limpio = nombre.replace(
        "numericas__",
        ""
    )

    limpio = limpio.replace(
        "categoricas__",
        ""
    )

    limpio = traducciones.get(
        limpio,
        limpio
    )

    nombres_limpios.append(
        limpio
    )

plt.figure(
    figsize=(24, 12)
)

plot_tree(
    arbol,
    feature_names=nombres_limpios,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=9
)

plt.title(
    "Árbol de Decisión de Regresión",
    fontsize=16
)

plt.tight_layout()
plt.show()


## ¿Cómo se lee el árbol?

Cada cuadro representa una decisión.

Ejemplo:

`Puntos local <= 20`

significa:

> “¿El equipo local tiene 20 puntos o menos?”

- Si la respuesta es **Sí**, seguimos por la izquierda.
- Si la respuesta es **No**, seguimos por la derecha.

Dentro de cada nodo también aparecerán valores como:

### `samples`

Cantidad de registros que llegaron a ese nodo.

### `value`

Promedio que el árbol utilizaría como predicción en ese nodo.

### `squared_error`

Indica qué tanta variación existe entre los valores del nodo.

Mientras menor sea, más parecidos son los registros agrupados allí.

### Para explicar

> “El árbol va haciendo preguntas sobre las estadísticas de los equipos. Según cada respuesta avanza por una rama diferente hasta llegar a una estimación del porcentaje de apoyo.”


# 16. Conclusiones automáticas

La siguiente celda utiliza los resultados reales obtenidos por tu modelo y genera conclusiones en puntos cortos.

Así evitamos inventar porcentajes o copiar resultados incorrectos.


In [ ]:
mse_final = resultado_optimo["MSE"]
r2_final = resultado_optimo["R2"]
mae_final = resultado_optimo["MAE"]

variable_mas_importante = (
    tabla_importancias.iloc[0]["Variable"]
)

print("CONCLUSIONES")
print("-" * 60)

print(
    f"• El Árbol de Decisión optimizado obtuvo "
    f"un MSE de {mse_final:.2f}."
)

print(
    f"• El modelo obtuvo un R² de "
    f"{r2_final:.3f}."
)

print(
    f"• El modelo se equivoca aproximadamente "
    f"{mae_final:.2f} puntos porcentuales en promedio."
)

if mse_optimo < mse_inicial:
    print(
        "• El ajuste de hiperparámetros redujo "
        "el error respecto al árbol inicial."
    )
else:
    print(
        "• El ajuste de hiperparámetros no redujo "
        "el error respecto al árbol inicial."
    )

print(
    f"• La variable con mayor importancia fue: "
    f"{variable_mas_importante}."
)

print(
    "• El Árbol de Decisión permite predecir "
    "un valor numérico y explicar qué variables "
    "influyen más en la estimación."
)


# 17. Conclusiones para la exposición

Después de ejecutar la celda anterior, puedes resumir los resultados así:

- El modelo predice el **porcentaje de apoyo al equipo local**.
- El **MSE** indica cuánto error tiene el modelo: menor es mejor.
- El **R²** indica cuánto logra explicar el modelo: mayor es mejor.
- El **MAE** permite decir aproximadamente cuántos puntos porcentuales se equivoca.
- Se comparó el árbol inicial con un árbol optimizado mediante validación cruzada.
- El modelo también permite identificar las variables con mayor importancia.

## Frase corta para cerrar

> “El Árbol de Decisión nos permitió estimar el porcentaje de apoyo al equipo local y, además, identificar qué estadísticas tienen mayor peso en la predicción.”

**Importante:** utiliza siempre los valores generados por el código y no inventes porcentajes.


# 18. Resumen rápido para estudiar

### ¿Qué estamos prediciendo?
El porcentaje de apoyo al equipo local.

### ¿Qué tipo de problema es?
Regresión, porque predecimos un número.

### ¿Qué modelo usamos?
`DecisionTreeRegressor`.

### ¿Cómo dividimos los datos?
80% entrenamiento y 20% prueba.

### ¿Qué hacemos con los nulos?
Mediana para números y moda para categorías.

### ¿Qué hace GridSearchCV?
Prueba diferentes configuraciones del árbol y selecciona la mejor.

### ¿Qué significa MSE?
Error cuadrático medio. Menor es mejor.

### ¿Qué significa R²?
Qué tanto de los datos puede explicar el modelo. Mayor es mejor.

### ¿Qué significa MAE?
Cuánto se equivoca aproximadamente el modelo en promedio.

### ¿Qué es feature importance?
Muestra qué variables utiliza más el árbol.

### ¿Qué significan los `(...)` del gráfico?
El árbol continúa, pero ocultamos niveles para poder leerlo.
